<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/%D0%9A%D0%BE%D1%80%D0%BF%D1%83%D1%81%D0%BD%D0%B8%D0%B9_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7%D0%B0%D1%82%D0%BE%D1%80_(VRT_%2B_POS).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- КРОК 1: ОНОВЛЕННЯ ТА ВСТАНОВЛЕННЯ ---
# Оновлюємо бібліотеку, щоб отримати доступ до тегів частин мови (POS)
!pip install --upgrade simplemma
!pip install python-docx openpyxl

import simplemma
import pandas as pd
from docx import Document
from google.colab import files
from collections import Counter
import re

# --- КРОК 2: КОНФІГУРАЦІЯ ТА ФІЛЬТРИ ---
LANGUAGES = ('en', 'uk', 'de')

# Стоп-слова для фільтрації n-грам (щоб фрази були змістовними)
STOP_WORDS = {
    'і', 'й', 'та', 'але', 'чи', 'або', 'що', 'як', 'це', 'на', 'в', 'у', 'до', 'за', 'з', 'із', 'зі',
    'він', 'вона', 'воно', 'вони', 'не', 'ні', 'так', 'the', 'a', 'an', 'and', 'or', 'of', 'to', 'be', 'have', 'in', 'on'
}

# Маркери шуму (фрази, що починаються/закінчуються цими словами, ігноруються)
NOISE_MARKERS = {
    'thursday', 'friday', 'monday', 'tuesday', 'wednesday', 'saturday', 'sunday',
    'today', 'yesterday', 'tomorrow', 'say', 'tell', 'talk', 'new', 'old', 'also'
}

def get_morphology(token, langs=LANGUAGES):
    """
    Безпечне отримання леми та POS-тегу. Сумісно з усіма версіями simplemma.
    """
    try:
        # Спроба отримати детальну інформацію
        res = simplemma.lemmatize(token, lang=langs, full=True)

        # Якщо результат - об'єкт (нова версія)
        if hasattr(res, 'lemma'):
            return res.lemma, getattr(res, 'pos', "UNKNOWN") or "UNKNOWN"

        # Якщо результат - кортеж (деякі версії)
        if isinstance(res, tuple):
            return res[0], res[1] if len(res) > 1 and res[1] else "UNKNOWN"

        return res, "UNKNOWN"
    except (TypeError, AttributeError):
        # Якщо розширене тегування не підтримується версією
        return simplemma.lemmatize(token, lang=langs), "UNKNOWN"

def is_valid_ngram(ngram_tuple):
    """ Перевірка лінгвістичної якості n-грами. """
    if not ngram_tuple: return False
    if ngram_tuple[0] in NOISE_MARKERS or ngram_tuple[-1] in NOISE_MARKERS:
        return False
    if any(word in STOP_WORDS for word in ngram_tuple):
        return False
    return True

def extract_ngrams(sentences, n=2):
    """ Генерація частотних списків n-грам. """
    res = []
    for s in sentences:
        if len(s) >= n:
            for i in range(len(s)-n+1):
                ngram = tuple(s[i:i+n])
                if is_valid_ngram(ngram):
                    res.append(" ".join(ngram))
    return res

def process_corpus_data(text, langs=LANGUAGES):
    """ Генерація вертикального тегованого тексту та даних для звіту. """
    sentences_raw = re.split(r'([.!?;\n]+)', text)
    annotated_tokens = []
    clean_sentences_for_ngrams = []
    vertical_lines = []

    for part in sentences_raw:
        if not part.strip(): continue

        # Обробка пунктуації (тег PUNCT)
        if any(char in '.!?;\n' for char in part):
            punc = part.strip()
            if punc: vertical_lines.append(f"{punc}\t{punc}\tPUNCT")
            continue

        tokens = simplemma.simple_tokenizer(part)
        current_sent_lemmas = []

        for token in tokens:
            clean_token = re.sub(r'[^\w\s]', '', token)

            if not clean_token or clean_token.isdigit():
                vertical_lines.append(f"{token}\t{token}\tNUM")
                continue

            lemma, pos = get_morphology(clean_token, langs)
            low_lemma = lemma.lower()

            # ВЕРТИКАЛЬНИЙ ФОРМАТ (VRT): Слово [TAB] Лема [TAB] Тег
            vertical_lines.append(f"{token}\t{low_lemma}\t{pos}")

            # Дані для статистики та Excel
            annotated_tokens.append({'Word': token, 'Lemma': low_lemma, 'POS': pos})

            # Тільки значущі леми для n-грам
            if low_lemma not in STOP_WORDS:
                current_sent_lemmas.append(low_lemma)

        if current_sent_lemmas:
            clean_sentences_for_ngrams.append(current_sent_lemmas)

    return pd.DataFrame(annotated_tokens), clean_sentences_for_ngrams, "\n".join(vertical_lines)

# --- КРОК 3: ЗАПУСК ---
print("Завантажте ваш текстовий файл (.txt):")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    input_text = uploaded[file_name].decode('utf-8', errors='ignore')

    # Обробка
    df_tokens, clean_sentences, v_text = process_corpus_data(input_text)

    # Аналіз
    bigrams = extract_ngrams(clean_sentences, n=2)
    trigrams = extract_ngrams(clean_sentences, n=3)
    bigram_freq = Counter(bigrams).most_common(40)
    trigram_freq = Counter(trigrams).most_common(40)

    if not df_tokens.empty:
        pos_stats = df_tokens['POS'].value_counts().reset_index()
        pos_stats.columns = ['Part of Speech', 'Count']
    else:
        pos_stats = pd.DataFrame(columns=['Part of Speech', 'Count'])

    # --- КРОК 4: ЕКСПОРТ ---

    # 1. ТЕГОВАНИЙ ТЕКСТ (VRT)
    vrt_file = 'corpus_tagged_vertical.vrt'
    with open(vrt_file, 'w', encoding='utf-8') as f:
        f.write(v_text)

    # 2. Excel
    excel_name = 'corpus_data.xlsx'
    with pd.ExcelWriter(excel_name) as writer:
        df_tokens.to_excel(writer, sheet_name='Annotation', index=False)
        pd.DataFrame(bigram_freq, columns=['Bigram', 'Freq']).to_excel(writer, sheet_name='Bigrams', index=False)
        pd.DataFrame(trigram_freq, columns=['Trigram', 'Freq']).to_excel(writer, sheet_name='Trigrams', index=False)

    # 3. Word
    word_name = 'corpus_report.docx'
    doc = Document()
    doc.add_heading('Корпусний аналіз: тегування та одиниці', 0)

    doc.add_heading('Розподіл частин мови', level=1)
    table = doc.add_table(rows=1, cols=2); table.style = 'Table Grid'
    table.rows[0].cells[0].text = 'POS-тег'; table.rows[0].cells[1].text = 'Кількість'
    for _, row in pos_stats.iterrows():
        r = table.add_row().cells
        r[0].text = str(row['Part of Speech']); r[1].text = str(row['Count'])

    def add_ng_table(title, data):
        doc.add_heading(title, level=1)
        t = doc.add_table(rows=1, cols=2); t.style = 'Table Grid'
        t.rows[0].cells[0].text = 'Одиниця'; t.rows[0].cells[1].text = 'Частота'
        for item, freq in data:
            r = t.add_row().cells
            r[0].text = item; r[1].text = str(freq)

    add_ng_table('Топ Біграми (n=2)', bigram_freq[:25])
    add_ng_table('Топ Триграми (n=3)', trigram_freq[:25])
    doc.save(word_name)

    print(f"\nФайли успішно створено:\n1. {vrt_file} (Тегований вертикальний текст)\n2. {excel_name} (Таблиці)\n3. {word_name} (Звіт)")

    # Автоматичне завантаження VRT файлу
    files.download(vrt_file)
else:
    print("Файл не обрано.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.7 MB/s eta 0:00:00
Завантажте ваш текстовий файл (.txt):


Saving zelensky_hails_new_ideas_on_peace_after_talk_with_.txt to zelensky_hails_new_ideas_on_peace_after_talk_with_.txt

Файли успішно створено:
1. corpus_tagged_vertical.vrt (Тегований вертикальний текст)
2. corpus_data.xlsx (Таблиці)
3. corpus_report.docx (Звіт)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>